# Red neuronal MLP para datos tabulares

Usa esta plantilla cuando pidan una red neuronal para datos tabulares.

Ejemplos: dataset CSV con columnas numéricas/categóricas y una etiqueta.

**Trampas principales:**
- `X` debe ser `float32`, `y` debe ser `long` (multiclase) o `float32` (binaria con BCELoss)
- `CrossEntropyLoss` espera logits sin Softmax — NO añadir Softmax al final
- El preprocesado (scaler, imputer) SIEMPRE se hace antes de crear tensors, con sklearn
- Usar `model.train()` antes del loop y `model.eval()` antes de evaluar

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# SIEMPRE fijar semillas para reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

# SIEMPRE definir device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

TARGET = 'target'  # CAMBIAR

In [ ]:
df = pd.read_csv('data.csv')
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [ ]:
# Preprocesado con sklearn — la red neuronal necesita números, sin NaN, escalados
numeric_cols    = X_train.select_dtypes(include='number').columns
categorical_cols = X_train.select_dtypes(exclude='number').columns

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ]), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        # sparse_output=False → devuelve array denso, no matriz sparse
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical_cols),
])

# fit SOLO en train, transform en ambos
X_train_np = preprocessor.fit_transform(X_train).astype(np.float32)
X_test_np  = preprocessor.transform(X_test).astype(np.float32)

# LabelEncoder para convertir etiquetas a integers 0,1,2,...
le = LabelEncoder()
y_train_np = le.fit_transform(y_train)
y_test_np  = le.transform(y_test)

print('Features tras preprocesar:', X_train_np.shape[1])
print('Clases:', le.classes_)

In [ ]:
# Convertir a tensors y mandar al device
X_train_t = torch.tensor(X_train_np).to(device)                          # float32
X_test_t  = torch.tensor(X_test_np).to(device)
y_train_t = torch.tensor(y_train_np, dtype=torch.long).to(device)        # long para CrossEntropyLoss
y_test_t  = torch.tensor(y_test_np,  dtype=torch.long).to(device)

# DataLoader — shuffle solo en train
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=64, shuffle=False)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size: int, num_classes: int) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),   # Dropout reduce overfitting — desactivado en eval()
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
            # NO softmax aquí — CrossEntropyLoss lo incluye
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


model = MLP(
    input_size=X_train_t.shape[1],
    num_classes=len(le.classes_)
).to(device)

# CrossEntropyLoss para multiclase (>2 clases O 2 clases con etiquetas long)
# Si es BINARIA con Sigmoid: usar BCEWithLogitsLoss y y como float, no long
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print('Modelo:', model)
print('Parámetros:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# Loop de entrenamiento con registro de train loss y test accuracy
EPOCHS = 30
history = {'train_loss': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    # --- TRAIN ---
    model.train()   # activa Dropout
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()           # 1. limpiar gradientes
        logits = model(X_batch)         # 2. forward
        loss   = criterion(logits, y_batch)  # 3. calcular error
        loss.backward()                 # 4. backprop
        optimizer.step()                # 5. actualizar pesos
        train_loss += loss.item()

    # --- EVAL ---
    model.eval()    # desactiva Dropout
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            preds   = model(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total   += y_batch.size(0)

    avg_loss = train_loss / len(train_loader)
    acc      = correct / total
    history['train_loss'].append(avg_loss)
    history['test_acc'].append(acc)

    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={avg_loss:.4f}  test_acc={acc:.4f}')

In [ ]:
# Curvas de entrenamiento — si train_loss baja pero test_acc no sube → overfitting
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history['train_loss']); axes[0].set_title('Train Loss');    axes[0].set_xlabel('Epoch')
axes[1].plot(history['test_acc']);   axes[1].set_title('Test Accuracy');  axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

In [ ]:
# Reporte final
model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        all_preds.extend(model(X_batch).argmax(dim=1).cpu().tolist())

print(classification_report(y_test_np, all_preds, target_names=le.classes_.astype(str), zero_division=0))

## Variante BINARIA (2 clases con BCEWithLogitsLoss)

Si el examen pide específicamente regresión logística como red neuronal, usa esta variante.

In [ ]:
# Diferencias respecto a multiclase:
# 1. Última capa: nn.Linear(64, 1)  — 1 neurona en vez de num_classes
# 2. Loss: BCEWithLogitsLoss()       — incluye Sigmoid internamente
# 3. y debe ser float32, no long
# 4. Predicción: (logits > 0).long() o (torch.sigmoid(logits) > 0.5).long()

class MLPBinary(nn.Module):
    def __init__(self, input_size: int) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),   # 1 sola neurona de salida
            # NO Sigmoid — BCEWithLogitsLoss lo incluye
        )

    def forward(self, x):
        return self.network(x)

# Uso:
# y_train_t_bin = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)  # [N, 1]
# criterion_bin = nn.BCEWithLogitsLoss()
# preds = (model_bin(X_test_t) > 0).long().squeeze()  # threshold en 0 (equivale a 0.5 en sigmoid)
print('Plantilla binaria lista')

## Si falla

- **Loss no baja**: revisa escalado, NaN, y que el learning rate no sea demasiado alto.
- **Error de tipos**: `X` debe ser `float32`, `y` debe ser `long` (multiclase) o `float32` (binaria).
- **Error de dimensiones en Linear**: el `input_size` no cuadra — imprime `X_train_t.shape[1]`.
- **Overfitting**: sube dropout, añade más datos, o simplifica la arquitectura (menos capas/neuronas).
- **RuntimeError CUDA**: baja `batch_size` o usa `device='cpu'`.